# 07 — Streaming Simulation — ShipTrack Week 10

**Project:** P10 ShipTrack — Supply Chain Visibility Hub | **Team:** 10
**Notebook path:** `notebooks/07_streaming_simulation.ipynb`
**Technology:** Databricks Free Edition (serverless) · Auto Loader · Structured Streaming · Spark SQL first

**Goal:** show ShipTrack scan events arriving over time, in two ways:

| Part | What it proves | Output |
|---|---|---|
| A — Auto Loader file drops | five controlled `scan_event.json` drops are ingested incrementally; the checkpoint stops old files being re-read | `bronze_scan_event_stream` |
| B — Controlled stream DQ | replay, sequence conflict, late event, orphan reference, lifecycle, schema drift and malformed lines are routed with evidence | `silver_candidate_scan_events`, `trusted_silver_scan_events`, `quarantine_scan_event_stream`, `gold_fact_scan_event` |
| C — Live scan events | a small producer creates events while a continuous SQL pipeline keeps processing them | `bronze_shiptrack_live_scan_events` |

**Boundary:** no Kafka installation, no Gold KPI redesign, no Power BI change. Kafka is design-only (`streaming/kafka_event_schema.json`).

> Run one cell at a time on the first run. Do **not** use *Run all*: the point of Part A is that files arrive one after another.

## 1. Source-pack facts (not Databricks results)

The approved P10 data pack supplies five newline-delimited JSON drops. These line counts come from `source_manifest.csv` in the pack.

| Drop | Physical lines | What it tests |
|---|---:|---|
| `drop_01_baseline` | 12 | normal valid events |
| `drop_02_duplicate_replay` | 14 | replayed `event_id` and a conflicting shipment sequence |
| `drop_03_late_out_of_order` | 14 | event time, ordering and the 24-hour watermark |
| `drop_04_reference_lifecycle` | 15 | orphan reference, wrong hub for the lane, invalid lifecycle |
| `drop_05_malformed_schema_drift` | 15 | malformed JSON, unsupported field, unsupported `schema_version` |
| **All drops** | **70** | |

### Upload once before running
The same files are in the repo under `data_sample/streaming/`. Upload them to the volume folder `/Volumes/workspace/default/shiptrack_stream/week10_source/` in **either** layout:

- `week10_source/drop_01_baseline/scan_event.json` (same folders as the pack), **or**
- `week10_source/drop_01_baseline.json` (file renamed to its drop name — easiest in the upload dialog).

Section 2 creates the volume and folders if they do not exist.

## 2. Configuration, folders and helpers

In [ ]:
import os, shutil
from pyspark.sql import functions as F

CATALOG, SCHEMA, VOLUME = "workspace", "default", "shiptrack_stream"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

BASE           = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SOURCE_DIR     = f"{BASE}/week10_source"       # untouched copies of the five drops
LANDING_DIR    = f"{BASE}/week10_landing"      # the only folder Auto Loader watches
CHECKPOINT_DIR = f"{BASE}/week10_checkpoint/bronze_scan_event_stream"   # separate from input and output
BRONZE_STREAM_TABLE = "bronze_scan_event_stream"

DROPS = ["drop_01_baseline", "drop_02_duplicate_replay", "drop_03_late_out_of_order",
         "drop_04_reference_lifecycle", "drop_05_malformed_schema_drift"]

for d in (SOURCE_DIR, LANDING_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

def find_drop(drop):
    """Return the uploaded source file for a drop, whichever upload layout was used."""
    for candidate in (f"{SOURCE_DIR}/{drop}/scan_event.json",
                      f"{SOURCE_DIR}/{drop}.json",
                      f"{SOURCE_DIR}/{drop}_scan_event.json"):
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f"{drop} not found under {SOURCE_DIR} - upload it first (see section 1).")

def land(drop):
    """Simulate a file arrival: copy ONE drop from source into the landing folder."""
    target = f"{LANDING_DIR}/{drop}.json"
    if os.path.exists(target):
        print(f"Already in landing (not copied again): {target}")
    else:
        shutil.copy(find_drop(drop), target)
        print(f"Landed: {target}")
    print("Landing now contains:", sorted(os.listdir(LANDING_DIR)))

print("Source     :", SOURCE_DIR)
print("Landing    :", LANDING_DIR)
print("Checkpoint :", CHECKPOINT_DIR)

### 2.1 Confirm the five source drops
Physical line counts must match the source-pack facts in section 1 (12, 14, 14, 15, 15).

In [ ]:
for drop in DROPS:
    path = find_drop(drop)
    with open(path, "r", encoding="utf-8") as fh:
        lines = [ln for ln in fh.read().splitlines() if ln.strip()]
    print(f"{drop:<34} physical_lines={len(lines):>3}   {path}")
print("Landing starts with:", sorted(os.listdir(LANDING_DIR)), "(expected: empty before Drop 01)")

### 2.2 Look at Drop 01 before streaming
One row = one scan event for one shipment at one event time.

In [ ]:
display(
    spark.read.json(find_drop("drop_01_baseline"))
         .select("source_record_id", "event_id", "shipment_id", "event_sequence_no", "shipment_status",
                 "event_time", "hub_id", "carrier_id", "route_id", "schema_version")
)

## 3. Event contract and Auto Loader source

- The schema is **declared explicitly** (21 fields from the pack `data_dictionary.csv`); nothing is inferred.
- Auto Loader reads each file **line by line** (`cloudFiles.format = text`) so every physical line is preserved in `raw_line`, including malformed ones. The line is then parsed with the explicit schema.
- Bronze does **not** deduplicate, repair or filter. Replays and bad lines must stay visible.

In [ ]:
EVENT_SCHEMA = (
    "source_record_id STRING, event_id STRING, shipment_id STRING, event_sequence_no INT, "
    "event_type STRING, shipment_status STRING, event_time TIMESTAMP, producer_time TIMESTAMP, "
    "hub_id STRING, carrier_id STRING, route_id STRING, origin_hub_id STRING, destination_hub_id STRING, "
    "service_level STRING, transport_mode STRING, exception_type STRING, package_condition STRING, "
    "source_system STRING, schema_version STRING, run_id STRING, status_reason STRING"
)
EXPECTED_FIELDS = [f.strip().split(" ")[0] for f in EVENT_SCHEMA.split(",")]
expected_arr = F.array(*[F.lit(c) for c in EXPECTED_FIELDS])

raw_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "text")
         .schema("value STRING")
         .load(LANDING_DIR)
         .select(F.col("value").alias("raw_line"),
                 F.col("_metadata.file_path").alias("_source_file"),
                 F.col("_metadata.file_name").alias("_source_file_name"))
         .where(F.length(F.trim(F.col("raw_line"))) > 0)
)

events_stream = (
    raw_stream
      .withColumn("p", F.from_json("raw_line", EVENT_SCHEMA))                 # explicit typed schema
      .withColumn("m", F.from_json("raw_line", "map<string,string>"))         # raw key/value view of the same line
      .withColumn("_is_malformed",
                  F.col("m").isNull()
                  | ~F.col("raw_line").rlike(r"^\s*\{.*\}\s*$")
                  | F.get_json_object("raw_line", "$").isNull())
      .select(
          "p.*",
          F.col("m")["event_sequence_no"].alias("_raw_event_sequence_no"),
          F.col("m")["event_time"].alias("_raw_event_time"),
          F.col("m")["producer_time"].alias("_raw_producer_time"),
          "_is_malformed",
          F.when(F.col("m").isNotNull(), F.array_except(F.map_keys("m"), expected_arr)).alias("_unexpected_fields"),
          F.when(F.col("m").isNotNull(), F.array_except(expected_arr, F.map_keys("m"))).alias("_missing_fields"),
          "raw_line", "_source_file", "_source_file_name",
          F.sha2(F.concat_ws("|", "_source_file_name", "raw_line"), 256).alias("_bronze_line_hash"),
          F.current_timestamp().alias("_ingested_at"),
      )
)
print("Auto Loader source is defined. Nothing has been written yet.")

### 3.1 One small streaming-write helper

Plain English: *process whatever is newly available, remember progress in the checkpoint, append to Bronze, stop when caught up.*
`availableNow` is the trigger supported on Databricks Free Edition serverless compute.

In [ ]:
def run_available_now():
    query = (
        events_stream.writeStream
            .option("checkpointLocation", CHECKPOINT_DIR)
            .trigger(availableNow=True)
            .toTable(BRONZE_STREAM_TABLE)
    )
    query.awaitTermination()
    print("AvailableNow run finished. Bronze rows now:", spark.table(BRONZE_STREAM_TABLE).count())

# Part A — Auto Loader: five drops arriving over time

## 4. Drop 01 — baseline

In [ ]:
land("drop_01_baseline")
run_available_now()

### 4.1 Per-file reconciliation
**Source baseline for Drop 01:** 12 physical lines.

In [ ]:
%sql
SELECT _source_file_name,
       COUNT(*)                                   AS physical_rows,
       COUNT(DISTINCT event_id)                   AS distinct_event_ids,
       SUM(CASE WHEN _is_malformed THEN 1 ELSE 0 END) AS malformed_lines,
       MIN(_ingested_at)                          AS ingested_at
FROM bronze_scan_event_stream
GROUP BY _source_file_name
ORDER BY _source_file_name;

### 4.2 What the checkpoint has recorded
Stronger evidence than saying "the checkpoint exists".

In [ ]:
%sql
SELECT *
FROM cloud_files_state('/Volumes/workspace/default/shiptrack_stream/week10_checkpoint/bronze_scan_event_stream')
ORDER BY path;

## 5. Drop 02 — duplicate replay
Both files are now physically in landing. Only the checkpoint stops Drop 01 becoming new work again.

In [ ]:
land("drop_02_duplicate_replay")
run_available_now()

In [ ]:
%sql
SELECT _source_file_name, COUNT(*) AS physical_rows, COUNT(DISTINCT event_id) AS distinct_event_ids
FROM bronze_scan_event_stream
GROUP BY _source_file_name
ORDER BY _source_file_name;

### 5.1 Replayed business events
A repeated `event_id` here is a **business replay inside a new file**, not the stream re-reading an old file. Bronze keeps both physical rows.

In [ ]:
%sql
SELECT event_id, source_record_id, shipment_id, event_sequence_no, shipment_status, event_time, _source_file_name
FROM bronze_scan_event_stream
WHERE event_id IN (SELECT event_id FROM bronze_scan_event_stream GROUP BY event_id HAVING COUNT(*) > 1)
ORDER BY event_id, source_record_id;

### 5.2 Conflicting shipment sequence
Two different `event_id` values claiming the same `(shipment_id, event_sequence_no)`.

In [ ]:
%sql
SELECT shipment_id, event_sequence_no, event_id, source_record_id, shipment_status, event_time
FROM bronze_scan_event_stream
WHERE (shipment_id, event_sequence_no) IN (
        SELECT shipment_id, event_sequence_no
        FROM bronze_scan_event_stream
        GROUP BY shipment_id, event_sequence_no
        HAVING COUNT(DISTINCT event_id) > 1)
ORDER BY shipment_id, event_sequence_no, event_id;

## 6. No-new-file test — checkpoint proof
Nothing new has arrived, so nothing new may be appended. `rows_before` must equal `rows_after`.

In [ ]:
%sql
SELECT COUNT(*) AS rows_before FROM bronze_scan_event_stream;

In [ ]:
run_available_now()   # no new file was landed before this run

In [ ]:
%sql
SELECT COUNT(*) AS rows_after FROM bronze_scan_event_stream;

## 7. Drop 03 — late and out-of-order events

In [ ]:
land("drop_03_late_out_of_order")
run_available_now()

## 8. Drop 04 — reference and lifecycle cases

In [ ]:
land("drop_04_reference_lifecycle")
run_available_now()

## 9. Drop 05 — malformed line and schema drift

In [ ]:
land("drop_05_malformed_schema_drift")
run_available_now()

### 9.1 Final per-file reconciliation
**Source baseline:** 12 / 14 / 14 / 15 / 15 physical lines = 70.

In [ ]:
%sql
SELECT COALESCE(_source_file_name, 'ALL FILES') AS source_file,
       COUNT(*)                                   AS physical_rows,
       COUNT(DISTINCT event_id)                   AS distinct_event_ids,
       SUM(CASE WHEN _is_malformed THEN 1 ELSE 0 END) AS malformed_lines
FROM bronze_scan_event_stream
GROUP BY ROLLUP(_source_file_name)
ORDER BY source_file;

### 9.2 Malformed and drifted lines are preserved, not dropped

In [ ]:
%sql
SELECT source_record_id, event_id, schema_version, _is_malformed, _unexpected_fields, _missing_fields,
       SUBSTR(raw_line, 1, 110) AS raw_line_start, _source_file_name
FROM bronze_scan_event_stream
WHERE _is_malformed OR SIZE(_unexpected_fields) > 0 OR SIZE(_missing_fields) > 0 OR COALESCE(schema_version, '') <> '1.0';

### 9.3 Event time vs ingestion time
`event_time` = when the scan happened. `producer_time` = when the source system emitted it. `_ingested_at` = when Databricks ingested it.

In [ ]:
%sql
SELECT event_id, shipment_status, event_time, producer_time, _ingested_at, _source_file_name
FROM bronze_scan_event_stream
ORDER BY _ingested_at, event_time
LIMIT 20;

### 9.4 Delta history and final checkpoint state
Use your own history as evidence. Do not type version numbers or timestamps by hand.

In [ ]:
%sql
DESCRIBE HISTORY bronze_scan_event_stream;

In [ ]:
%sql
SELECT *
FROM cloud_files_state('/Volumes/workspace/default/shiptrack_stream/week10_checkpoint/bronze_scan_event_stream')
ORDER BY path;

# Part B — Controlled stream DQ: Candidate → Trusted / Quarantine → Gold

Bronze kept every physical line. Part B applies the approved ShipTrack rules to the streamed events.

| Rule | Severity | Streamed event fails when |
|---|---|---|
| `DQ-EVT-001` | Critical / Major | line is malformed JSON; a required field is missing or unparseable; `schema_version` is not `1.0`; an undeclared field arrives (schema drift); the `event_id` was already received (replay — first arrival is kept); `event_time` is older than the 24-hour watermark |
| `DQ-REF-001` | Critical | shipment, hub, carrier or route is not in the trusted references, or the event lane does not match the route lane |
| `DQ-SCN-001` | Major | two different `event_id` values claim the same `(shipment_id, event_sequence_no)` (both rows fail — no arbitrary winner); or the sequence is non-positive |
| `DQ-TIM-001` | Major | `producer_time` is earlier than `event_time` |
| `DQ-DEL-001` | Major | a `DELIVERED` event is not at the lane destination hub; or the producer marks the transition `INVALID_AFTER_DELIVERY` |

**Watermark.** Watermark = (latest `event_time` seen in earlier arrivals) − 24 hours. Each event records `watermark_status` (`ON_TIME`, `LATE_WITHIN_WATERMARK`, `OUTSIDE_WATERMARK`) and `is_late`. Late-but-inside events stay trusted with the flag; outside-watermark events go to Quarantine.

**Every failure is kept.** One quarantine row carries the full `failed_rule_list`; no check short-circuits another.

## 10. Reference handoff
Streaming events are validated against the Week-6 trusted references. The cell prints exactly which table was used for each reference.

In [ ]:
REF_CANDIDATES = {
    "shipments": ["trusted_silver_shipments", "trusted_shipments"],
    "hubs":      ["trusted_silver_hubs", "trusted_hubs", "silver_hubs"],
    "carriers":  ["trusted_silver_carriers", "trusted_carriers", "silver_carriers"],
    "routes":    ["trusted_silver_routes", "trusted_routes", "silver_routes"],
}
def first_existing(names):
    for n in names:
        if spark.catalog.tableExists(n):
            return n
    return None

resolved = {k: first_existing(v) for k, v in REF_CANDIDATES.items()}
for k, v in resolved.items():
    note = "" if (v or "").startswith("trusted") else "   <-- WARNING: not a Trusted table; record this in the Week-10 log"
    print(f"  {k:<10} -> {v}{note}")
missing = [k for k, v in resolved.items() if v is None]
if missing:
    raise RuntimeError(f"Reference table(s) not found: {missing}. Run the Week-6 notebook first.")

norm = lambda c: F.upper(F.trim(F.col(c)))
spark.table(resolved["shipments"]).select(norm("shipment_id").alias("shipment_id")).distinct().createOrReplaceTempView("ref_shipments")
spark.table(resolved["hubs"]).select(norm("hub_id").alias("hub_id")).distinct().createOrReplaceTempView("ref_hubs")
spark.table(resolved["carriers"]).select(norm("carrier_id").alias("carrier_id")).distinct().createOrReplaceTempView("ref_carriers")
routes = spark.table(resolved["routes"])
routes.select(norm("route_id").alias("route_id")).distinct().createOrReplaceTempView("ref_routes")
routes.select(norm("route_id").alias("route_id"), norm("origin_hub_id").alias("origin_hub_id"),
              norm("destination_hub_id").alias("destination_hub_id")).distinct().createOrReplaceTempView("ref_route_lanes")
print("Reference views ready: ref_shipments, ref_hubs, ref_carriers, ref_routes, ref_route_lanes")

## 11. B1 — Silver Candidate (typed, standardised, one row per Bronze line)
Nothing is filtered or deduplicated here. A malformed line has no usable `source_record_id`, so it gets the key `UNPARSED-<line hash>` and stays countable.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_scan_events USING DELTA AS
WITH b AS (
  SELECT *, DENSE_RANK() OVER (ORDER BY _ingested_at) AS arrival_batch_no
  FROM bronze_scan_event_stream
),
batch_max AS (
  SELECT arrival_batch_no, MAX(event_time) AS batch_max_event_time
  FROM b GROUP BY arrival_batch_no
),
wm AS (
  SELECT arrival_batch_no,
         MAX(batch_max_event_time) OVER (ORDER BY arrival_batch_no
                                         ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_event_time_seen_before
  FROM batch_max
)
SELECT
  COALESCE(NULLIF(TRIM(b.source_record_id), ''),
           CONCAT('UNPARSED-', SUBSTR(b._bronze_line_hash, 1, 16)))  AS source_record_id,
  NULLIF(UPPER(TRIM(b.event_id)), '')            AS event_id,
  NULLIF(UPPER(TRIM(b.shipment_id)), '')         AS shipment_id,
  b.event_sequence_no,
  NULLIF(UPPER(TRIM(b.event_type)), '')          AS event_type,
  NULLIF(UPPER(TRIM(b.shipment_status)), '')     AS shipment_status,
  b.event_time,
  CAST(b.event_time AS DATE)                     AS event_date,
  b.producer_time,
  NULLIF(UPPER(TRIM(b.hub_id)), '')              AS hub_id,
  NULLIF(UPPER(TRIM(b.carrier_id)), '')          AS carrier_id,
  NULLIF(UPPER(TRIM(b.route_id)), '')            AS route_id,
  NULLIF(UPPER(TRIM(b.origin_hub_id)), '')       AS origin_hub_id,
  NULLIF(UPPER(TRIM(b.destination_hub_id)), '')  AS destination_hub_id,
  NULLIF(UPPER(TRIM(b.service_level)), '')       AS service_level,
  NULLIF(UPPER(TRIM(b.transport_mode)), '')      AS transport_mode,
  NULLIF(UPPER(TRIM(b.exception_type)), '')      AS exception_type,
  NULLIF(UPPER(TRIM(b.package_condition)), '')   AS package_condition,
  NULLIF(TRIM(b.source_system), '')              AS source_system,
  NULLIF(TRIM(b.schema_version), '')             AS schema_version,
  NULLIF(TRIM(b.run_id), '')                     AS run_id,
  NULLIF(UPPER(TRIM(b.status_reason)), '')       AS status_reason,
  b._raw_event_sequence_no, b._raw_event_time, b._raw_producer_time,
  b._is_malformed, b._unexpected_fields, b._missing_fields,
  b.arrival_batch_no,
  wm.max_event_time_seen_before,
  wm.max_event_time_seen_before - INTERVAL 24 HOURS AS watermark_ts,
  CASE
    WHEN b.event_time IS NULL THEN 'NO_EVENT_TIME'
    WHEN wm.max_event_time_seen_before IS NULL OR b.event_time >= wm.max_event_time_seen_before THEN 'ON_TIME'
    WHEN b.event_time >= wm.max_event_time_seen_before - INTERVAL 24 HOURS THEN 'LATE_WITHIN_WATERMARK'
    ELSE 'OUTSIDE_WATERMARK'
  END                                            AS watermark_status,
  COALESCE(b.event_time < wm.max_event_time_seen_before, FALSE) AS is_late,
  b.raw_line, b._source_file, b._source_file_name, b._bronze_line_hash, b._ingested_at,
  current_timestamp()                            AS _candidate_built_at
FROM b
LEFT JOIN wm ON b.arrival_batch_no = wm.arrival_batch_no;

In [ ]:
%sql
SELECT (SELECT COUNT(*) FROM bronze_scan_event_stream)      AS bronze_physical_rows,
       (SELECT COUNT(*) FROM silver_candidate_scan_events)  AS candidate_rows,
       (SELECT COUNT(DISTINCT source_record_id) FROM silver_candidate_scan_events) AS candidate_distinct_source_record_id;

## 12. B2 — Rule results at the physical record key
Every rule is evaluated independently. `failed_rule_list` keeps all of them.

In [ ]:
%sql
CREATE OR REPLACE TABLE stream_scan_event_dq_results USING DELTA AS
WITH c AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY event_id
                            ORDER BY arrival_batch_no, _source_file_name, source_record_id) AS event_id_arrival_rank
  FROM silver_candidate_scan_events
),
seq AS (
  SELECT shipment_id, event_sequence_no, COUNT(DISTINCT event_id) AS event_ids_on_sequence
  FROM c
  WHERE shipment_id IS NOT NULL AND event_sequence_no IS NOT NULL AND event_id IS NOT NULL
  GROUP BY shipment_id, event_sequence_no
),
f AS (
  SELECT c.*,
    COALESCE(c._is_malformed, FALSE)                                                        AS e_malformed,
    COALESCE(NOT c._is_malformed AND (c.source_record_id LIKE 'UNPARSED-%' OR c.event_id IS NULL OR c.shipment_id IS NULL
             OR c.event_sequence_no IS NULL OR c.event_type IS NULL OR c.shipment_status IS NULL OR c.event_time IS NULL
             OR c.hub_id IS NULL OR c.carrier_id IS NULL OR c.route_id IS NULL), FALSE)     AS e_required,
    COALESCE(NOT c._is_malformed AND COALESCE(c.schema_version, '') <> '1.0', FALSE)        AS e_version,
    COALESCE(NOT c._is_malformed AND SIZE(c._unexpected_fields) > 0, FALSE)                 AS e_drift,
    COALESCE(c.event_id IS NOT NULL AND c.event_id_arrival_rank > 1, FALSE)                 AS e_replay,
    COALESCE(c.watermark_status = 'OUTSIDE_WATERMARK', FALSE)                               AS e_watermark,
    COALESCE(c.shipment_id IS NOT NULL AND rs.shipment_id IS NULL, FALSE)                   AS r_shipment,
    COALESCE(c.hub_id      IS NOT NULL AND rh.hub_id      IS NULL, FALSE)                   AS r_hub,
    COALESCE(c.carrier_id  IS NOT NULL AND rc.carrier_id  IS NULL, FALSE)                   AS r_carrier,
    COALESCE(c.route_id    IS NOT NULL AND rr.route_id    IS NULL, FALSE)                   AS r_route,
    COALESCE(rr.route_id IS NOT NULL AND c.origin_hub_id IS NOT NULL AND c.destination_hub_id IS NOT NULL
             AND rl.route_id IS NULL, FALSE)                                                AS r_lane,
    COALESCE(seq.event_ids_on_sequence > 1, FALSE)                                          AS s_conflict,
    COALESCE(c.event_sequence_no < 1, FALSE)                                                AS s_nonpositive,
    COALESCE(c.producer_time < c.event_time, FALSE)                                         AS t_producer,
    COALESCE(c.shipment_status = 'DELIVERED' AND c.hub_id <> c.destination_hub_id, FALSE)   AS d_hub,
    COALESCE(c.status_reason = 'INVALID_AFTER_DELIVERY', FALSE)                             AS d_lifecycle
  FROM c
  LEFT JOIN seq             ON c.shipment_id = seq.shipment_id AND c.event_sequence_no = seq.event_sequence_no
  LEFT JOIN ref_shipments rs ON c.shipment_id = rs.shipment_id
  LEFT JOIN ref_hubs      rh ON c.hub_id      = rh.hub_id
  LEFT JOIN ref_carriers  rc ON c.carrier_id  = rc.carrier_id
  LEFT JOIN ref_routes    rr ON c.route_id    = rr.route_id
  LEFT JOIN ref_route_lanes rl ON c.route_id = rl.route_id
                              AND c.origin_hub_id = rl.origin_hub_id
                              AND c.destination_hub_id = rl.destination_hub_id
),
g AS (
  SELECT f.*,
    (e_malformed OR e_required OR e_version OR e_drift OR e_replay OR e_watermark) AS fail_evt,
    (r_shipment OR r_hub OR r_carrier OR r_route OR r_lane)                        AS fail_ref,
    (s_conflict OR s_nonpositive)                                                  AS fail_scn,
    t_producer                                                                     AS fail_tim,
    (d_hub OR d_lifecycle)                                                         AS fail_del
  FROM f
)
SELECT g.*,
  FILTER(ARRAY(CASE WHEN fail_evt THEN 'DQ-EVT-001' END,
               CASE WHEN fail_ref THEN 'DQ-REF-001' END,
               CASE WHEN fail_scn THEN 'DQ-SCN-001' END,
               CASE WHEN fail_tim THEN 'DQ-TIM-001' END,
               CASE WHEN fail_del THEN 'DQ-DEL-001' END), x -> x IS NOT NULL)      AS failed_rule_list,
  CASE WHEN fail_evt OR fail_ref OR fail_scn OR fail_tim OR fail_del THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN e_malformed OR e_required OR e_version OR fail_ref THEN 'CRITICAL'
       WHEN fail_evt OR fail_scn OR fail_tim OR fail_del THEN 'MAJOR' END          AS severity,
  CONCAT_WS('; ',
    CASE WHEN e_malformed   THEN 'malformed JSON line' END,
    CASE WHEN e_required    THEN 'required event field missing or unparseable' END,
    CASE WHEN e_version     THEN CONCAT('unsupported schema_version ', COALESCE(schema_version, 'NULL'), ' (expected 1.0)') END,
    CASE WHEN e_drift       THEN CONCAT('undeclared field(s): ', ARRAY_JOIN(_unexpected_fields, ',')) END,
    CASE WHEN e_replay      THEN 'replayed event_id; first arrival kept' END,
    CASE WHEN e_watermark   THEN 'event_time older than 24-hour watermark' END,
    CASE WHEN r_shipment    THEN 'shipment_id not in trusted shipments' END,
    CASE WHEN r_hub         THEN 'hub_id not in hub reference' END,
    CASE WHEN r_carrier     THEN 'carrier_id not in carrier reference' END,
    CASE WHEN r_route       THEN 'route_id not in route reference' END,
    CASE WHEN r_lane        THEN 'event lane does not match route lane' END,
    CASE WHEN s_conflict    THEN 'conflicting event_id values on same shipment sequence' END,
    CASE WHEN s_nonpositive THEN 'non-positive event_sequence_no' END,
    CASE WHEN t_producer    THEN 'producer_time earlier than event_time' END,
    CASE WHEN d_hub         THEN 'DELIVERED event not at lane destination hub' END,
    CASE WHEN d_lifecycle   THEN 'producer marked transition INVALID_AFTER_DELIVERY' END) AS failure_reason,
  CONCAT_WS(',',
    CASE WHEN e_malformed   THEN 'raw_line' END,
    CASE WHEN e_required    THEN 'required_fields' END,
    CASE WHEN e_version     THEN 'schema_version' END,
    CASE WHEN e_drift       THEN '_unexpected_fields' END,
    CASE WHEN e_replay      THEN 'event_id' END,
    CASE WHEN e_watermark   THEN 'event_time' END,
    CASE WHEN r_shipment    THEN 'shipment_id' END,
    CASE WHEN r_hub         THEN 'hub_id' END,
    CASE WHEN r_carrier     THEN 'carrier_id' END,
    CASE WHEN r_route OR r_lane THEN 'route_id' END,
    CASE WHEN fail_scn      THEN 'event_sequence_no' END,
    CASE WHEN t_producer    THEN 'producer_time' END,
    CASE WHEN d_hub         THEN 'hub_id' END,
    CASE WHEN d_lifecycle   THEN 'status_reason' END)                              AS affected_field,
  current_timestamp()                                                              AS dq_checked_at,
  'SHIPTRACK-STREAM-W10-V1.0'                                                      AS dq_ruleset_version
FROM g;

## 13. B3 — Route once: Trusted or Quarantine

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_scan_events USING DELTA AS
SELECT source_record_id, event_id, shipment_id, event_sequence_no, event_type, shipment_status,
       event_time, event_date, producer_time, hub_id, carrier_id, route_id, origin_hub_id, destination_hub_id,
       service_level, transport_mode, exception_type, package_condition, status_reason,
       source_system, schema_version, run_id,
       watermark_status, is_late, arrival_batch_no,
       dq_status, dq_ruleset_version, dq_checked_at,
       _source_file_name, _bronze_line_hash, _ingested_at
FROM stream_scan_event_dq_results
WHERE dq_status = 'PASS';

In [ ]:
%sql
CREATE OR REPLACE TABLE quarantine_scan_event_stream USING DELTA AS
SELECT source_record_id, event_id, shipment_id, event_sequence_no, event_type, shipment_status,
       event_time, producer_time, hub_id, carrier_id, route_id, origin_hub_id, destination_hub_id,
       schema_version, status_reason, watermark_status, is_late, watermark_ts, arrival_batch_no,
       dq_status, failed_rule_list, SIZE(failed_rule_list) AS failed_rule_count,
       severity, failure_reason, affected_field,
       raw_line, _unexpected_fields, _missing_fields,
       _source_file, _source_file_name, _bronze_line_hash, _ingested_at, run_id,
       dq_ruleset_version, dq_checked_at,
       current_timestamp() AS quarantined_at,
       'OPEN'              AS rework_status
FROM stream_scan_event_dq_results
WHERE dq_status = 'FAIL';

## 14. B4 — Gold: `gold_fact_scan_event`
**Grain:** one trusted streaming event (`event_id`). Reads Trusted Silver only.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_fact_scan_event USING DELTA AS
SELECT event_id, shipment_id, event_sequence_no, event_type, shipment_status,
       event_time, event_date, hub_id, carrier_id, route_id, service_level, transport_mode,
       watermark_status, is_late, source_record_id,
       current_timestamp() AS gold_loaded_at
FROM trusted_silver_scan_events;

## 15. Proof
### 15.1 Reconciliation — Candidate = Trusted + Quarantine
Expected: `variance = 0`, `trusted_quarantine_overlap = 0`, `gold_duplicate_event_ids = 0`.

In [ ]:
%sql
SELECT c.candidate_distinct, t.trusted_distinct, q.quarantine_distinct,
       c.candidate_distinct - t.trusted_distinct - q.quarantine_distinct AS variance,
       o.trusted_quarantine_overlap,
       g.gold_rows, g.gold_rows - g.gold_distinct_event_ids AS gold_duplicate_event_ids,
       CASE WHEN c.candidate_distinct = t.trusted_distinct + q.quarantine_distinct
             AND o.trusted_quarantine_overlap = 0 AND g.gold_rows = g.gold_distinct_event_ids
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_result
FROM (SELECT COUNT(DISTINCT source_record_id) AS candidate_distinct FROM silver_candidate_scan_events) c
CROSS JOIN (SELECT COUNT(DISTINCT source_record_id) AS trusted_distinct FROM trusted_silver_scan_events) t
CROSS JOIN (SELECT COUNT(DISTINCT source_record_id) AS quarantine_distinct FROM quarantine_scan_event_stream) q
CROSS JOIN (SELECT COUNT(*) AS trusted_quarantine_overlap
            FROM trusted_silver_scan_events a JOIN quarantine_scan_event_stream b ON a.source_record_id = b.source_record_id) o
CROSS JOIN (SELECT COUNT(*) AS gold_rows, COUNT(DISTINCT event_id) AS gold_distinct_event_ids FROM gold_fact_scan_event) g;

### 15.2 Rule scorecard
Rule counts can overlap: one quarantined event may fail more than one rule.

In [ ]:
%sql
SELECT rule_id, SUM(failed) AS failed_events, COUNT(*) - SUM(failed) AS passed_events
FROM (
  SELECT 'DQ-EVT-001' AS rule_id, CASE WHEN fail_evt THEN 1 ELSE 0 END AS failed FROM stream_scan_event_dq_results
  UNION ALL SELECT 'DQ-REF-001', CASE WHEN fail_ref THEN 1 ELSE 0 END FROM stream_scan_event_dq_results
  UNION ALL SELECT 'DQ-SCN-001', CASE WHEN fail_scn THEN 1 ELSE 0 END FROM stream_scan_event_dq_results
  UNION ALL SELECT 'DQ-TIM-001', CASE WHEN fail_tim THEN 1 ELSE 0 END FROM stream_scan_event_dq_results
  UNION ALL SELECT 'DQ-DEL-001', CASE WHEN fail_del THEN 1 ELSE 0 END FROM stream_scan_event_dq_results
) r
GROUP BY rule_id
ORDER BY rule_id;

### 15.3 Quarantine evidence — every failed event with all its reasons

In [ ]:
%sql
SELECT source_record_id, event_id, shipment_id, failed_rule_list, severity, failure_reason, affected_field,
       _source_file_name, rework_status
FROM quarantine_scan_event_stream
ORDER BY _source_file_name, source_record_id;

### 15.4 Watermark evidence — late events

In [ ]:
%sql
SELECT watermark_status, dq_status, COUNT(*) AS events
FROM stream_scan_event_dq_results
GROUP BY watermark_status, dq_status
ORDER BY watermark_status, dq_status;

In [ ]:
%sql
SELECT source_record_id, event_id, event_time, max_event_time_seen_before, watermark_ts, watermark_status, is_late, dq_status
FROM stream_scan_event_dq_results
WHERE watermark_status IN ('OUTSIDE_WATERMARK')
   OR (watermark_status = 'LATE_WITHIN_WATERMARK' AND event_time < max_event_time_seen_before - INTERVAL 12 HOURS)
ORDER BY event_time;

### 15.5 Live metric — trusted scan events by hub and status
This is the near-real-time metric: it changes each time a new drop is processed and Part B is rebuilt.

In [ ]:
%sql
SELECT hub_id, shipment_status, COUNT(*) AS trusted_events,
       SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_events,
       MAX(event_time) AS latest_event_time
FROM gold_fact_scan_event
GROUP BY hub_id, shipment_status
ORDER BY trusted_events DESC, hub_id, shipment_status;

## 16. No-new-file rerun — zero incremental trusted events

1. Run 16.1 (stores the *before* counts).
2. Run 16.2 (stream runs again with **no** new file).
3. Re-run sections **11 → 14** (B1–B4).
4. Run 16.3 — every `diff` must be 0.

In [ ]:
%sql
-- 16.1 before
CREATE OR REPLACE TABLE week10_rerun_proof USING DELTA AS
SELECT 'before_rerun' AS stage, current_timestamp() AS captured_at,
       (SELECT COUNT(*) FROM bronze_scan_event_stream)     AS bronze_rows,
       (SELECT COUNT(*) FROM trusted_silver_scan_events)   AS trusted_rows,
       (SELECT COUNT(*) FROM quarantine_scan_event_stream)       AS quarantine_rows,
       (SELECT COUNT(*) FROM gold_fact_scan_event)         AS gold_rows;

In [ ]:
run_available_now()   # 16.2 - no new file landed

In [ ]:
%sql
-- 16.3 after (run only after re-running sections 11-14)
INSERT INTO week10_rerun_proof
SELECT 'after_rerun', current_timestamp(),
       (SELECT COUNT(*) FROM bronze_scan_event_stream),
       (SELECT COUNT(*) FROM trusted_silver_scan_events),
       (SELECT COUNT(*) FROM quarantine_scan_event_stream),
       (SELECT COUNT(*) FROM gold_fact_scan_event);

In [ ]:
%sql
SELECT a.bronze_rows AS bronze_before, b.bronze_rows AS bronze_after, b.bronze_rows - a.bronze_rows AS bronze_diff,
       a.trusted_rows AS trusted_before, b.trusted_rows AS trusted_after, b.trusted_rows - a.trusted_rows AS trusted_diff,
       b.quarantine_rows - a.quarantine_rows AS quarantine_diff, b.gold_rows - a.gold_rows AS gold_diff
FROM (SELECT * FROM week10_rerun_proof WHERE stage = 'before_rerun') a
CROSS JOIN (SELECT * FROM week10_rerun_proof WHERE stage = 'after_rerun' ORDER BY captured_at DESC LIMIT 1) b;

# Part C — Live scan events

Part A showed *files* arriving. Part C shows *business events* arriving while nobody uploads anything.

```text
small producer (this notebook)  ->  shiptrack_live_scan_events (Delta source)
        ->  continuous SQL pipeline: FROM STREAM(...)  ->  bronze_shiptrack_live_scan_events
        ->  SQL observation queries
```

Each live shipment moves through `PICKED_UP → IN_TRANSIT → AT_HUB → OUT_FOR_DELIVERY → DELIVERED`.
Live IDs start with `LIVE-` so they can never be confused with batch or drop data, and they are not loaded into Gold.

## 17. Create / reset the live source table

In [ ]:
%sql
CREATE OR REPLACE TABLE shiptrack_live_scan_events (
    event_id           STRING,
    shipment_id        STRING,
    event_sequence_no  INT,
    event_type         STRING,
    shipment_status    STRING,
    event_time         TIMESTAMP,
    hub_id             STRING,
    carrier_id         STRING,
    schema_version     STRING
) USING DELTA;

## 18. Start the continuous SQL pipeline (done once, in the Databricks UI)

*Jobs & Pipelines → Create → ETL pipeline* — Serverless **ON**, Pipeline mode **Continuous**, catalog `workspace`, schema `default`. Use this as the only transformation file, then **Start**:

```sql
CREATE OR REFRESH STREAMING TABLE workspace.default.bronze_shiptrack_live_scan_events
COMMENT 'ShipTrack Week 10 live scan events'
AS
SELECT event_id, shipment_id, event_sequence_no, event_type, shipment_status,
       event_time, hub_id, carrier_id, schema_version,
       current_timestamp() AS _ingested_at
FROM STREAM(workspace.default.shiptrack_live_scan_events);
```

The pipeline stays running. It is **not** re-run for every event.

## 19. Producer — run 1 (events 1–15)
One event every 3 seconds. Keep this cell running and use the queries in section 20 from a second tab.

In [ ]:
import time

LIVE_HUBS     = ["H001", "H003", "H004", "H006", "H009", "H011"]
LIVE_CARRIERS = ["C001", "C002", "C003"]
LIFECYCLE = [("PICKED_UP", "PICKED_UP"), ("DEPARTED_HUB", "IN_TRANSIT"), ("ARRIVED_HUB", "AT_HUB"),
             ("STATUS_UPDATE", "OUT_FOR_DELIVERY"), ("DELIVERY_CONFIRMATION", "DELIVERED")]

def produce(first_event, last_event, pause_seconds=3):
    """Create live scan events first_event..last_event (1-30). 30 events = 6 shipments x 5 lifecycle steps."""
    for i in range(first_event - 1, last_event):
        shipment_no = i // len(LIFECYCLE) + 1
        seq         = i % len(LIFECYCLE) + 1
        event_type, status = LIFECYCLE[seq - 1]
        hub     = LIVE_HUBS[(shipment_no - 1) % len(LIVE_HUBS)]
        carrier = LIVE_CARRIERS[(shipment_no - 1) % len(LIVE_CARRIERS)]
        spark.sql(
            "INSERT INTO shiptrack_live_scan_events VALUES "
            f"('LIVE-EVT-{i + 1:04d}', 'LIVE-SHP-{shipment_no:04d}', {seq}, '{event_type}', '{status}', "
            f"current_timestamp(), '{hub}', '{carrier}', '1.0')"
        )
        print(f"LIVE-EVT-{i + 1:04d}  LIVE-SHP-{shipment_no:04d}  seq={seq}  {status:<17} {hub}")
        time.sleep(pause_seconds)
    print("Producer stopped.")

produce(1, 15)

## 20. Observe the stream (run these repeatedly)
### 20.1 Newest live events

In [ ]:
%sql
SELECT event_time, shipment_id, event_sequence_no, shipment_status, hub_id, _ingested_at
FROM bronze_shiptrack_live_scan_events
ORDER BY event_time DESC
LIMIT 15;

### 20.2 Count growth
Run, wait about 10 seconds, run again. While the producer runs, the count rises.

In [ ]:
%sql
SELECT COUNT(*) AS events_received, MAX(event_time) AS latest_event_time, current_timestamp() AS observed_at
FROM bronze_shiptrack_live_scan_events;

### 20.3 Live metric — events by hub and status

In [ ]:
%sql
SELECT hub_id, shipment_status, COUNT(*) AS event_count
FROM bronze_shiptrack_live_scan_events
GROUP BY hub_id, shipment_status
ORDER BY hub_id, event_count DESC;

### 20.4 One shipment lifecycle in order

In [ ]:
%sql
SELECT shipment_id, event_sequence_no, event_type, shipment_status, event_time, hub_id
FROM bronze_shiptrack_live_scan_events
ORDER BY shipment_id, event_sequence_no
LIMIT 30;

## 21. Stop / restart proof

1. When producer run 1 has printed `Producer stopped.`, run 20.2 **twice**, 10 seconds apart — the count must stay the same.
2. Run the cell below (producer run 2) and run 20.2 again — the count must rise again.

```text
Producer ON  -> events arrive -> Bronze grows
Producer OFF -> no events     -> Bronze stays unchanged
```

In [ ]:
produce(16, 30)   # producer run 2

# Close

## 22. Evidence to capture (save in `screenshots/` with these names)

| File | Cell | What it proves |
|---|---|---|
| `week10_01_autoloader_source_drops.png` | 2.1 | five source drops with their line counts |
| `week10_02_autoloader_incremental.png` | 5 (per-file query after Drop 02) | new file changed Bronze incrementally |
| `week10_03_autoloader_reconciliation.png` | 9.1 | per-file and total physical / distinct counts |
| `week10_04_no_new_file_bronze.png` | 6 | `rows_before = rows_after` |
| `week10_05_stream_dq_reconciliation.png` | 15.1 | Candidate = Trusted + Quarantine, overlap 0 |
| `week10_06_quarantine_evidence.png` | 15.3 | every failed event with all rule IDs and reasons |
| `week10_07_watermark_late_events.png` | 15.4 | watermark status and late events |
| `week10_08_live_metric.png` | 15.5 | trusted events by hub and status |
| `week10_09_no_new_file_rerun.png` | 16.3 | all diffs 0 after a rerun |
| `week10_10_continuous_pipeline.png` | pipeline UI | pipeline running in continuous mode |
| `week10_11_live_events.png` | 20.1 | recent live events |
| `week10_12_count_growth.png` | 20.2 (two runs) | count higher on the second run |
| `week10_13_stop_restart.png` | 21 | stable count when stopped, growth after restart |

Record the actual numbers from these cells in `weekly_logs/week10_log.md`. Never type an expected value as a result.

## 23. Controlled reset (only before a completely fresh demonstration)
The source files in `week10_source/` are not touched.

In [ ]:
RESET = False   # set to True, run once, then set back to False
if RESET:
    for t in ["bronze_scan_event_stream", "silver_candidate_scan_events", "stream_scan_event_dq_results",
              "trusted_silver_scan_events", "quarantine_scan_event_stream", "gold_fact_scan_event", "week10_rerun_proof"]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    for d in (LANDING_DIR, CHECKPOINT_DIR):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    print("Week-10 stream tables, landing folder and checkpoint were reset.")
else:
    print("Reset skipped.")

## 24. Exit checklist

- [ ] Five drops landed one at a time; per-file counts match the source-pack facts
- [ ] Replay and sequence conflict shown in Bronze (5.1, 5.2) and routed in Part B
- [ ] No-new-file run added no Bronze rows (6) and no trusted events (16)
- [ ] Late, orphan, lifecycle, schema-drift and malformed events are in `quarantine_scan_event_stream` with reasons
- [ ] Candidate = Trusted + Quarantine; overlap = 0; `gold_fact_scan_event` has one row per `event_id`
- [ ] Continuous pipeline ran while the producer created events; count grew, stopped, and grew again
- [ ] Screenshots saved; `streaming/` docs and Week-10 log updated with real results

## 25. Quick viva

1. What does Auto Loader watch, and what does the checkpoint remember?
2. Why is a repeated `event_id` in Drop 02 not a checkpoint problem?
3. Why does Bronze keep the malformed line?
4. How is the 24-hour watermark calculated here, and what happens to an event outside it?
5. Why do both rows of a sequence conflict go to Quarantine?
6. Why does the live count stop changing when the producer stops?
7. Why is Kafka design-only in this project?